# 1 - Config & Import

## 1.1 - Config

In [1]:
# 🔧 config import
import os
from core.config.logger_config import colored_logger
logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-12-01 21:55:21] INFO - 726159153.py - Logger initialized (Notebook)


## 1.2 - Import

In [2]:
# 📦 External libs
import importlib
import pandas as pd

# 📂 Internal modules
import core.data.fetcher as fetcher
import core.data.features as features
import core.data.labels as labels

symbol = 'BTCUSDT'
start_date = '2025-06-25'
end_date = "2025-06-27"
interval = '1min'

def reload_modules():
    importlib.reload(features)
    importlib.reload(labels)
    importlib.reload(fetcher)

    Fetcher = fetcher.DataFetcher(symbol, start_date, end_date, interval)
    Fetcher.load_from_csv_dom_bybit(directory=os.getcwd())
    Features = features.Features(Fetcher.raw_data)
    return Features

Features = reload_modules()
print(Features.data.head())

[2025-12-01 22:07:22] INFO - fetcher.py - Logger initialized (fetcher.py)
[2025-12-01 22:07:22] INFO - features.py - Logger initialized (features.py)
[2025-12-01 22:07:22] INFO - labels.py - Logger initialized (labels.py)
[2025-12-01 22:07:22] INFO - features.py - Logger initialized (features.py)
[2025-12-01 22:07:22] INFO - labels.py - Logger initialized (labels.py)
[2025-12-01 22:07:22] INFO - fetcher.py - Logger initialized (fetcher.py)
[2025-12-01 22:07:22] INFO - fetcher.py - 📥 Data loaded from: C:\Users\Valérian Darmenté\PycharmProjects\Trading_Project\src\pipeline/BTCUSDT_2025-06-25_2025-06-27_1min_dom.csv


                 time      open      high       low     close  ask_size  \
0 2025-06-25 00:00:00  106087.9  106130.0  106072.0  106072.1  3.270138   
1 2025-06-25 00:01:00  106072.0  106097.2  106063.0  106071.8  1.049900   
2 2025-06-25 00:02:00  106071.9  106071.9  106022.8  106022.9  2.298718   
3 2025-06-25 00:03:00  106022.8  106079.7  106015.3  106065.8  1.608724   
4 2025-06-25 00:04:00  106065.8  106083.0  106041.0  106041.0  1.684092   

   bid_size  
0  4.575117  
1  0.560414  
2  0.002983  
3  2.392039  
4  1.465795  


# 2 - Features

## Variables

In [3]:
# Resample with vwap
resample_period = '5min'

# Pivot Points
pivot_left = 15
pivot_right = 15

# Volume Pivot Points
duration_min = 240
n_cross = 7
std_factor = 1.0

# Std
std_window = 10

# Mean
mean_window = 10

## Process

In [4]:
Features = reload_modules()
Features.data["volume"] = Features.data["bid_size"] + Features.data["ask_size"]
print(Features.data.shape)
print(Features.data.columns)
print(Features.data.head())

# Step 1: Resample with VWAP (Liquidity)
Features.resample_with_vwap(resample_period)

# Step 2: Market Sessions (Market Context)
Features.market_sessions()

# Step 3: Pivot Points (Support/Resistance)
Features.Pivot_Points(pivot_left, pivot_right)

# Step 4: Volume Pivot Points (Volume-based S/R)
Features.volume_Pivot_Points(duration_min, n_cross, std_factor)

# Step 5: Volume Delta (Momentum)
Features.add_volume_delta()

# Step 6: Cumulative Volume Delta - CVD (Momentum)
Features.add_cvd()

# Step 7: Order Book Imbalance - OBI (Pressure)
Features.add_obi()

# Step 8: Price Change (Price Action)
Features.add_price_change()

# Step 9: Reaction Ratio (Liquidity)
Features.add_reaction_ratio()

# Step 10: Rolling Standard Deviation of Price (Volatility)
Features.add_rolling_std_price(std_window)

# Step 11: Rolling Mean of CVD (Smoothed Flow)
Features.add_rolling_mean_cvd(mean_window)

Features.data["volume"] = Features.data["bid_size"] + Features.data["ask_size"]
print(Features.data.shape)
print(Features.data.columns)
print(Features.data.head())


[2025-12-01 22:08:45] INFO - features.py - Logger initialized (features.py)
[2025-12-01 22:08:45] INFO - labels.py - Logger initialized (labels.py)
[2025-12-01 22:08:45] INFO - fetcher.py - Logger initialized (fetcher.py)
[2025-12-01 22:08:45] INFO - fetcher.py - 📥 Data loaded from: C:\Users\Valérian Darmenté\PycharmProjects\Trading_Project\src\pipeline/BTCUSDT_2025-06-25_2025-06-27_1min_dom.csv
[2025-12-01 22:08:45] INFO - features.py - Starting resampling with period: 5min
[2025-12-01 22:08:45] INFO - features.py - Basic OHLCV resampling done.
[2025-12-01 22:08:45] INFO - features.py - VWAP calculation completed.
[2025-12-01 22:08:45] INFO - features.py - Resampling successful. Resulting shape: (864, 8)
[2025-12-01 22:08:45] INFO - features.py - Starting market session flag generation.
[2025-12-01 22:08:45] INFO - features.py - Datetime index successfully converted and localized.
[2025-12-01 22:08:45] INFO - features.py - Market session flags successfully added.
[2025-12-01 22:08:45]

(4320, 8)
Index(['time', 'open', 'high', 'low', 'close', 'ask_size', 'bid_size',
       'volume'],
      dtype='object')
                 time      open      high       low     close  ask_size  \
0 2025-06-25 00:00:00  106087.9  106130.0  106072.0  106072.1  3.270138   
1 2025-06-25 00:01:00  106072.0  106097.2  106063.0  106071.8  1.049900   
2 2025-06-25 00:02:00  106071.9  106071.9  106022.8  106022.9  2.298718   
3 2025-06-25 00:03:00  106022.8  106079.7  106015.3  106065.8  1.608724   
4 2025-06-25 00:04:00  106065.8  106083.0  106041.0  106041.0  1.684092   

   bid_size    volume  
0  4.575117  7.845255  
1  0.560414  1.610314  
2  0.002983  2.301701  
3  2.392039  4.000763  
4  1.465795  3.149887  


[2025-12-01 22:08:45] INFO - features.py - Initial pivot marking complete.
[2025-12-01 22:08:46] INFO - features.py - Pivot filtering logic applied (highs/lows lists, broken levels).
[2025-12-01 22:08:46] INFO - features.py - Pivot point detection completed successfully.
[2025-12-01 22:08:46] INFO - features.py - Starting volume_Pivot_Points with duration_min=240, n_cross=7, std_factor=1.0
[2025-12-01 22:08:46] INFO - features.py - VWAP and speed computed.
[2025-12-01 22:08:46] INFO - features.py - Detected 25 upward and 26 downward VWAP crosses.
[2025-12-01 22:08:46] INFO - features.py - Calculated rolling pivot boundaries and last cross prices.
[2025-12-01 22:08:46] INFO - features.py - volume pivot point detection completed successfully.
[2025-12-01 22:08:46] INFO - features.py - Adding 'volume_delta' column...
[2025-12-01 22:08:46] INFO - features.py - 'volume_delta' column added.
[2025-12-01 22:08:46] INFO - features.py - Adding 'CVD' column...
[2025-12-01 22:08:46] INFO - feature

(864, 27)
Index(['open', 'high', 'low', 'close', 'volume', 'bid_size', 'ask_size',
       'VWAP_5m', 'london_open', 'ny_open', 'hk_open', 'dif_low_pivot',
       'dif_high_pivot', 'low_pivot', 'high_pivot', 'Rolling_VWAP_240min',
       'volume_low_Pivot', 'volume_high_Pivot', 'Dif_volume_low_Pivot',
       'Dif_volume_high_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd'],
      dtype='object')
                         open      high       low     close     volume  \
time                                                                     
2025-06-25 00:00:00  106087.9  106130.0  106015.3  106041.0  18.907920   
2025-06-25 00:05:00  106041.0  106218.3  106034.1  106218.3  27.845968   
2025-06-25 00:10:00  106218.3  106454.5  106149.1  106440.7  60.408618   
2025-06-25 00:15:00  106440.7  106441.7  106160.2  106267.9  40.931766   
2025-06-25 00:20:00  106267.9  106268.0  106101.2  106132.3  22.556447   

            

In [ ]:
import pandas as pd
from IPython.display import display
# Set option to display all columns
pd.set_option('display.max_columns', None)
# Display the DataFrame
pd.set_option('display.max_rows', None)
display(Features.data)

In [6]:
pd.reset_option('display.max_columns')
pd.reset_option('display.max_rows')

## Visualization

In [8]:
import asyncio
from lightweight_charts import Chart
import pandas as pd

df = Features.data.reset_index()
print(df.columns[1:])
# Close any previous chart
try:
    chart.exit()
except NameError:
    pass

# Create chart
chart = Chart(toolbox=True, maximize=True)
chart.legend(visible=True, font_size=14)
chart.set(df[['time','open','high','low','close','volume']])
#--------------------------------------------------
#--------------------------------------------------
# Create a color dictionary for each indicator.
indicator_colors = {
    'VWAP_5m': '#FFD700',              # Gold (very visible on black)
    'low_pivot': '#00BFFF',            # DeepSkyBlue
    'high_pivot': '#FF4500',           # OrangeRed (opposite of blue)
    'volume_low_Pivot': '#00FF7F',     # SpringGreen
    'volume_high_Pivot': '#FF1493',    # DeepPink (opposite of green)
    'Rolling_VWAP_240min': '#FFFFFF'   # White
}

for column in indicator_colors.keys():
    chart.create_line(
        name=column,
        color=indicator_colors[column],
        style="solid",
        width=1,
        price_line=False,
        price_label=True
    ).set(df[['time', column]])


#--------------------------------------------------
#--------------------------------------------------
# Display chart asynchronously in Jupyter
async def display():
    await chart.show_async()

asyncio.get_event_loop().create_task(display())

Index(['open', 'high', 'low', 'close', 'volume', 'bid_size', 'ask_size',
       'VWAP_5m', 'london_open', 'ny_open', 'hk_open', 'dif_low_pivot',
       'dif_high_pivot', 'low_pivot', 'high_pivot', 'Rolling_VWAP_240min',
       'volume_low_Pivot', 'volume_high_Pivot', 'Dif_volume_low_Pivot',
       'Dif_volume_high_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd'],
      dtype='object')


<Task pending name='Task-57' coro=<display() running at C:\Users\Valérian Darmenté\AppData\Local\Temp\ipykernel_46844\3528751609.py:43>>

# 3 - Label Data

## Variables

In [9]:
# Step 1: Categorize Pivot Points
look_forward=20

# Step 2: Categorize Volume Pivot Points
look_forward=20

## Process

In [10]:
reload_modules()

# Step 0: Initialize Labels Dataframe
Labels_Data = labels.Labels(Features.data)

# Step 1: Categorize Pivot Points
Labels_Data.categorize_pivot_points(look_forward)

# Step 2: Categorize Volume Pivot Points
Labels_Data.categorize_colume_pivot_points(look_forward)

#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Labels_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Labels_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

[2025-12-01 22:21:47] INFO - features.py - Logger initialized (features.py)
[2025-12-01 22:21:47] INFO - labels.py - Logger initialized (labels.py)
[2025-12-01 22:21:47] INFO - fetcher.py - Logger initialized (fetcher.py)
[2025-12-01 22:21:47] INFO - fetcher.py - 📥 Data loaded from: C:\Users\Valérian Darmenté\PycharmProjects\Trading_Project\src\pipeline/BTCUSDT_2025-06-25_2025-06-27_1min_dom.csv
[2025-12-01 22:21:47] INFO - labels.py - Starting Categorize_Pivot_Points with look_forward=20
[2025-12-01 22:21:47] INFO - labels.py - Categorize_Pivot_Points completed successfully.
[2025-12-01 22:21:47] INFO - labels.py - Starting Categorize_Volume_Pivot_Points with look_forward=20
[2025-12-01 22:21:48] INFO - labels.py - Categorize_Volume_Pivot_Points completed successfully.


#----------------------------------------------------------------------
Shape: (864, 31)
#----------------------------------------------------------------------
Index(['open', 'high', 'low', 'close', 'volume', 'bid_size', 'ask_size',
       'VWAP_5m', 'london_open', 'ny_open', 'hk_open', 'dif_low_pivot',
       'dif_high_pivot', 'low_pivot', 'high_pivot', 'Rolling_VWAP_240min',
       'volume_low_Pivot', 'volume_high_Pivot', 'Dif_volume_low_Pivot',
       'Dif_volume_high_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd',
       'low_below_pivot', 'high_above_pivot', 'vwap_below_volume_low',
       'vwap_above_volume_high'],
      dtype='object')
#----------------------------------------------------------------------
                         open      high       low     close     volume  \
time                                                                     
2025-06-25 00:00:00  106087.9  106130.0  106015.3  10604

In [ ]:
import pandas as pd
from IPython.display import display
# Set option to display all columns
pd.set_option('display.max_columns', None)
# Display the DataFrame
pd.set_option('display.max_rows', None)
display(Labels_Data.data)

In [90]:
pd.reset_option('display.max_columns')
pd.reset_option('display.max_rows')

## Visualization

In [11]:
import asyncio
from lightweight_charts import Chart
import pandas as pd

df = Labels_Data.data.reset_index()
print(df.columns[1:])
# Close any previous chart
try:
    chart.exit()
except NameError:
    pass

# Create chart
chart = Chart(toolbox=True, maximize=True)
chart.legend(visible=True, font_size=14)
chart.set(df[['time','open','high','low','close','volume']])
#--------------------------------------------------
#--------------------------------------------------
# Create a color dictionary for each indicator.
indicator_colors = {
    'VWAP_5m': '#FFD700',              # Gold (very visible on black)
    'low_pivot': '#00BFFF',            # DeepSkyBlue
    'high_pivot': '#FF4500',           # OrangeRed (opposite of blue)
    'volume_low_Pivot': '#00FF7F',     # SpringGreen
    'volume_high_Pivot': '#FF1493',    # DeepPink (opposite of green)
    'Rolling_VWAP_240min': '#FFFFFF'   # White
}

for column in indicator_colors.keys():
    chart.create_line(
        name=column,
        color=indicator_colors[column],
        style="solid",
        width=1,
        price_line=False,
        price_label=True
    ).set(df[['time', column]])

#--------------------------------------------------
#--------------------------------------------------
# Columns for markers
column_marker = {
    'low_below_pivot': '#00BFFF',            # DeepSkyBlue
    'high_above_pivot': '#FF4500',           # OrangeRed (opposite of blue)
    'vwap_below_volume_low': '#00FF7F',      # SpringGreen
    'vwap_above_volume_high': '#FF1493',     # DeepPink (opposite of gree
}

# Prepare markers for each condition
markers = []
print(column_marker.keys())
for col in column_marker.keys():
    subset = df[df[col] == 1]
    position = 'below' if 'below' in col else 'above'
    color = column_marker[col]
    shape = 'arrowUp' if 'below' in col else 'arrowDown'
    text = ""

    for _, row in subset.iterrows():
        markers.append({
            'time': row['time'],
            'position': position,
            'color': color,
            'shape': shape,
            'text': text
        })
# Add markers to chart (bulk)
chart.marker_list(markers)

#--------------------------------------------------
#--------------------------------------------------
# Display chart asynchronously in Jupyter
async def display():
    await chart.show_async()

asyncio.get_event_loop().create_task(display())

Index(['open', 'high', 'low', 'close', 'volume', 'bid_size', 'ask_size',
       'VWAP_5m', 'london_open', 'ny_open', 'hk_open', 'dif_low_pivot',
       'dif_high_pivot', 'low_pivot', 'high_pivot', 'Rolling_VWAP_240min',
       'volume_low_Pivot', 'volume_high_Pivot', 'Dif_volume_low_Pivot',
       'Dif_volume_high_Pivot', 'volume_delta', 'CVD', 'obi', 'price_change',
       'reaction_ratio', 'rolling_std_price', 'rolling_mean_cvd',
       'low_below_pivot', 'high_above_pivot', 'vwap_below_volume_low',
       'vwap_above_volume_high'],
      dtype='object')
dict_keys(['low_below_pivot', 'high_above_pivot', 'vwap_below_volume_low', 'vwap_above_volume_high'])


<Task pending name='Task-76' coro=<display() running at C:\Users\Valérian Darmenté\AppData\Local\Temp\ipykernel_46844\3178861444.py:73>>

# 4 - Save

In [103]:
Labels_Data.data.to_csv("features_labels_data.csv", index=False)